# UC-PROC-1 — List and Execute a Process via OGC API - Processes

**Persona:** Platform Operator

**Goal:** Discover processes available on the platform, inspect the description
of one process, execute it via `POST /processes/{id}/execution`, poll the
resulting job to completion, and retrieve results. Finish with a deep link
to the processes browser web page.

**Prerequisites:**
- GeoID platform running at `DYNASTORE_BASE_URL` (default `http://localhost:8080`)
- `processes` extension enabled in the active SCOPE
- Write-scoped JWT in `DYNASTORE_TOKEN` (or IDP client credentials set)
- At least one process registered in the deployment

**References:**
- OGC API - Processes 1.0 (OGC 18-062r2)
- Routes: `/processes/processes`, `/processes/processes/{id}/execution`, `/processes/jobs/{job_id}`

In [ ]:
import json
import os
import time

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision token via IDP client_credentials when not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass

TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_WRITE_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)

# Override PROCESS_ID with a process id registered in your deployment
PROCESS_ID = os.environ.get("PROCESS_ID", "")

headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

print(f"Platform  : {BASE_URL}")
print(f"Process ID: {PROCESS_ID or '(will auto-select first available)'}")

## Step 1 — List available processes

`GET /processes/processes` returns all processes registered in the deployment.
Each entry includes the process id, title, and available execution typologies
(sync, async, cloud_run, etc.).

In [ ]:
r = client.get("/processes/processes")
print(f"status: {r.status_code}")

_processes_active = r.status_code == 200
_selected_process_id = PROCESS_ID

if _processes_active:
    payload = r.json()
    procs = payload.get("processes", [])
    print(f"Processes registered: {len(procs)}")
    for p in procs:
        pid = p.get("id", "")
        title = p.get("title", "")
        typologies = p.get("typologies", [])
        print(f"  id={pid!r}  title={title!r}  typologies={[t.get('runner_type') for t in typologies]}")
        if not _selected_process_id:
            _selected_process_id = pid
    if _selected_process_id:
        print(f"\nUsing process: {_selected_process_id!r}")
    else:
        print("\nNo processes found — check that at least one process is registered.")
elif r.status_code == 404:
    print("SKIP: /processes/processes not found — processes extension may not be active.")
else:
    print(r.text[:300])

## Step 2 — Inspect process description

`GET /processes/processes/{process_id}` returns the full OGC process description:
inputs schema, outputs schema, and execution links.

In [ ]:
if not _processes_active or not _selected_process_id:
    print("SKIP: processes extension not active or no process available.")
else:
    r = client.get(f"/processes/processes/{_selected_process_id}")
    print(f"process description status: {r.status_code}")
    if r.status_code == 200:
        desc = r.json()
        print(f"  id         : {desc.get('id')}")
        print(f"  title      : {desc.get('title')}")
        print(f"  description: {str(desc.get('description', ''))[:120]}")
        inputs_keys = list((desc.get("inputs") or {}).keys())
        print(f"  inputs     : {inputs_keys}")
        outputs_keys = list((desc.get("outputs") or {}).keys())
        print(f"  outputs    : {outputs_keys}")
    elif r.status_code == 404:
        print(f"  404: process '{_selected_process_id}' not found.")
    else:
        print(r.text[:300])

## Step 3 — Execute the process (async job)

`POST /processes/processes/{process_id}/execution` submits an execution request.
When the server returns 201 with a `jobID`, the job runs asynchronously. We
include `Prefer: respond-async` to request asynchronous execution explicitly.

The `dry_run` input is supported by most platform-scope processes and completes
without performing any writes — safe to call in any environment.

In [ ]:
_job_id = None

if not _processes_active or not _selected_process_id:
    print("SKIP: processes extension not active or no process available.")
else:
    exec_headers = {
        "Authorization": f"Bearer {TOKEN}",
        "Content-Type": "application/json",
        "Prefer": "respond-async",
    }
    exec_client = httpx.Client(base_url=BASE_URL, headers=exec_headers, timeout=60.0)

    exec_payload = {
        "inputs": {"dry_run": True},
        "response": "document",
    }

    r = exec_client.post(
        f"/processes/processes/{_selected_process_id}/execution",
        content=json.dumps(exec_payload),
    )
    print(f"execution status: {r.status_code}")

    if r.status_code in (200, 201):
        status_info = r.json()
        _job_id = status_info.get("jobID")
        print(f"  jobID  : {_job_id}")
        print(f"  status : {status_info.get('status')}")
        print(f"  message: {status_info.get('message', '')}")
    elif r.status_code == 400:
        print(f"  400: {r.text[:300]}")
        print("  Hint: the selected process may not support 'dry_run' — set PROCESS_ID to a valid process id.")
    elif r.status_code in (401, 403):
        print(f"  {r.status_code}: authentication or authorization required.")
    elif r.status_code == 404:
        print(f"  404: process '{_selected_process_id}' not found or not executable at this path.")
    else:
        print(r.text[:300])
    exec_client.close()

## Step 4 — Poll job status

`GET /processes/jobs/{job_id}` returns the current `StatusInfo` for a submitted
job. We poll until the status leaves the `running` / `accepted` states or the
timeout is exceeded.

In [ ]:
if not _job_id:
    print("SKIP: no job to poll (execution step did not return a jobID).")
else:
    MAX_POLLS = 10
    POLL_INTERVAL_S = 2
    final_status = None

    for attempt in range(MAX_POLLS):
        r = client.get(f"/processes/jobs/{_job_id}")
        if r.status_code != 200:
            print(f"  poll {attempt + 1}: {r.status_code} {r.text[:200]}")
            break
        info = r.json()
        st = info.get("status", "")
        print(f"  poll {attempt + 1}: status={st}  message={info.get('message', '')!r}")
        if st not in ("accepted", "running"):
            final_status = st
            break
        time.sleep(POLL_INTERVAL_S)

    if final_status:
        print(f"\nFinal status: {final_status}")
    else:
        print("\nJob still running after max polls — check the processes browser.")

## Step 5 — Retrieve job results

`GET /processes/jobs/{job_id}/results` returns the outputs of a completed job.

In [ ]:
if not _job_id:
    print("SKIP: no job to retrieve results for.")
else:
    r = client.get(f"/processes/jobs/{_job_id}/results")
    print(f"results status: {r.status_code}")
    if r.status_code == 200:
        results = r.json()
        print(json.dumps(results, indent=2)[:500])
    elif r.status_code == 404:
        print("  404: job not found or results not yet available.")
    else:
        print(r.text[:300])

## Result — Open the processes browser

In [ ]:
client.close()
print(f"Open the processes browser: {BASE_URL}/web/pages/processes_browser?language=en")
if _job_id:
    print(f"Job status URL: {BASE_URL}/processes/jobs/{_job_id}")